# Fitting-factor configuration features

This notebook demonstrates fixed-candidate fitting factors, generator-backed fitting factors, selected parameter optimization, and nested comparison configuration.

In [ ]:
import numpy as np

import waveformtools.comparison  # installs ModesArray.fitting_factor
from waveformtools.comparison import AlignmentSpec, FittingFactorConfig, ModeComparisonConfig, RotationSpec, fixed_candidate_fitting_factor
from waveformtools.modes_array import ModesArray


def make_modes(phase=0.0, amplitude=1.0, time_axis=None, orbital_phase=0.0):
    if time_axis is None:
        time_axis = np.linspace(-10.0, 10.0, 256)
    envelope = np.exp(-0.05 * time_axis**2)
    carrier = np.exp(1j * 0.7 * time_axis)
    modes = ModesArray(ell_max=2, time_axis=time_axis, spin_weight=-2)
    modes.create_modes_array(ell_max=2, data_len=len(time_axis))
    modes.set_mode_data(ell=2, emm=2, data=amplitude * envelope * carrier * np.exp(1j * phase) * np.exp(2j * orbital_phase))
    modes.set_mode_data(ell=2, emm=-2, data=0.3 * amplitude * envelope * np.conjugate(carrier) * np.exp(1j * phase) * np.exp(-2j * orbital_phase))
    return modes


target = make_modes(phase=0.35, amplitude=1.2)


## Fixed-candidate fitting factor

In [ ]:
candidate = make_modes(phase=0.35, amplitude=1.2)

comparison = ModeComparisonConfig(
    alignment=AlignmentSpec(time_alignment="none", phase_alignment="global_complex")
)
result = fixed_candidate_fitting_factor(target, candidate, config=comparison)

print("match=", result.match)
print("mismatch=", result.mismatch)
print("alignment=", result.best_parameters["alignment"])


## Generator-backed fitting factor with model defaults

In [ ]:
def generator(phase=0.35, amplitude=1.2):
    return make_modes(phase=phase, amplitude=amplitude)


result = target.fitting_factor(
    generator,
    config=FittingFactorConfig(
        comparison=ModeComparisonConfig(
            alignment=AlignmentSpec(time_alignment="none", phase_alignment="global_complex")
        ),
        fixed_parameters={},
        optimizer="none",
    ),
)

print("match=", result.match)
print("waveform generations=", result.n_waveform_generations)
print("generator parameters=", result.best_parameters["generator"])


## Optimizing selected generator parameters

In [ ]:
result = target.fitting_factor(
    generator,
    config=FittingFactorConfig(
        comparison=ModeComparisonConfig(
            alignment=AlignmentSpec(time_alignment="none", phase_alignment="none")
        ),
        variable_parameters={"phase": (-1.0, 1.0), "amplitude": (0.8, 1.5)},
        initial_parameters={"phase": 0.0, "amplitude": 1.0},
        optimizer="scipy_minimize",
        optimizer_options={"options": {"maxiter": 80}},
    ),
)

print("match=", result.match)
print("mismatch=", result.mismatch)
print("best generator parameters=", result.best_parameters["generator"])
print("optimizer diagnostics=", result.diagnostics["optimizer_diagnostics"])


## Combining generator optimization with alignment and rotation settings

In [ ]:
rotated_target = make_modes(phase=0.35, amplitude=1.2, orbital_phase=0.2)

result = rotated_target.fitting_factor(
    generator,
    config=FittingFactorConfig(
        comparison=ModeComparisonConfig(
            alignment=AlignmentSpec(time_alignment="none", phase_alignment="global_complex"),
            rotation=RotationSpec(kind="z_axis", optimize_angle=True, angle_bounds=(0.0, 0.5)),
        ),
        variable_parameters={"phase": (-1.0, 1.0)},
        initial_parameters={"phase": 0.0},
        optimizer="scipy_minimize",
        optimizer_options={"options": {"maxiter": 80}},
    ),
)

print("match=", result.match)
print("best generator parameters=", result.best_parameters["generator"])
print("best alignment parameters=", result.best_parameters["alignment"])
